# Multiview DICOM header analysis
Mean age by Index / Pre-Index / Non-Cancer.

In [16]:
import pickle
import pandas as pd

In [17]:
HEADER_PKL = "/home/a.kanamarlapudi001/projects/omama-proj/avanith/original_data/HQ/study_level_dicom_header/2d_studylevel_all_dicom_headers.pkl"
headers = pickle.load(open(HEADER_PKL, "rb"))

In [18]:
# PatientAge example: "687M" -> 687 months -> 57.25 years
study_age = {}

for ds in headers.values():
    study_id = str(ds.StudyInstanceUID)

    # already have this study
    if study_id in study_age:
        continue

    age_text = str(ds.PatientAge).strip()   # e.g. "687M"
    if not age_text.endswith("M"):
        continue

    months = int(age_text.replace("M", ""))
    years = months / 12
    study_age[study_id] = years

In [19]:
# same labels as 256 metadata (we already checked)
LABEL_CSV = "/home/a.kanamarlapudi001/projects/omama-proj/avanith/original_data/HQ/deepsight/manifests/final_curated_list_v3_3way_labels.csv"
labels = pd.read_csv(LABEL_CSV)
study_label = dict(zip(labels["study_id"], labels["label"]))

In [20]:
ages = {
    "IndexCancer": [],
    "PreIndexCancer": [],
    "NonCancer": [],
}

for study_id, label in study_label.items():
    if study_id not in study_age:
        continue
    ages[label].append(study_age[study_id])

for label, age_list in ages.items():
    mean_age = sum(age_list) / len(age_list)
    print(label, len(age_list), round(mean_age, 2))

IndexCancer 1220 56.33
PreIndexCancer 618 55.7
NonCancer 1997 51.53
